# Assignment TREND

This notebook implements four trend systems:
- 10/30 moving average crossover
- 30/100 moving average crossover
- 80/160 moving average crossover
- 30 day breakout

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.4f}'.format)

## Load the data

In [2]:
def load_sheet(sheet_name):
    df = pd.read_excel(
        '/Users/jlaw/projects/stern/systematic-investing/data/assignment_TREND_data.xlsx',
        sheet_name=sheet_name,
        header=1
    ).copy()

    # Clean the data
    df = df[df['#Date'].notna() & (df['#Date'] > 0)].copy()
    df['Date'] = pd.to_datetime(df['#Date'].astype(int).astype(str), format='%Y%m%d')

    # Choose desired date range
    df = df[df['Date'] <= pd.Timestamp('2010-12-31')].copy()

    # Get returns
    df['ret'] = df['PC(%)'] / 100
    df['index_close'] = 100 * (1 + df['ret']).cumprod()
    
    # Breakout levels
    df['index_high'] = df['index_close'] * (df['High'] / df['Close'])
    df['index_low'] = df['index_close'] * (df['Low'] / df['Close'])

    return df.reset_index(drop=True)


uro = load_sheet('uro')
ty = load_sheet('ty')
sp = load_sheet('sp')

uro[['Date', 'Close', 'PC(%)', 'index_close']].head()

,Date,Close,PC(%),index_close
0,1999-03-08,1.0943,0.0000,100.0000
1,1999-03-09,1.0944,0.0091,100.0091
2,1999-03-10,1.1006,0.5665,100.5757
3,1999-03-11,1.1084,0.7087,101.2885
4,1999-03-12,1.0957,-1.1458,100.1279


## Strategy rules

For moving averages, compare short with the long moving average.

For 30 day breakout, compare today's close with the highest high and lowest low over past 30 days.

Shift the trading signal by one day before multiplying by returns, so strategy does not use same-day information.

In [3]:
def sharpe_ratio(returns):
    clean = returns.dropna()
    return np.sqrt(252) * clean.mean() / clean.std()


def max_drawdown(performance, dates):
    running_max = performance.cummax()
    drawdown = performance / running_max - 1
    trough_index = drawdown.idxmin()
    peak_index = performance.loc[:trough_index].idxmax()

    return drawdown.loc[trough_index], dates.loc[peak_index], dates.loc[trough_index]


def moving_average_strategy(df, short_window, long_window):
    result = df[['Date', 'ret', 'index_close']].copy()
    result['short_ma'] = result['index_close'].rolling(short_window).mean()
    result['long_ma'] = result['index_close'].rolling(long_window).mean()

    result['signal'] = 0.0
    result.loc[result['short_ma'] > result['long_ma'], 'signal'] = 1.0
    result.loc[result['short_ma'] < result['long_ma'], 'signal'] = -1.0
    result.loc[result[['short_ma', 'long_ma']].isna().any(axis=1), 'signal'] = np.nan

    result['strategy_ret'] = result['signal'].shift(1) * result['ret']
    result['equity'] = (1 + result['strategy_ret'].fillna(0)).cumprod()

    return result


def breakout_strategy(df, window=30):
    result = df[['Date', 'ret', 'index_close', 'index_high', 'index_low']].copy()
    result['hi30'] = result['index_high'].shift(1).rolling(window).max()
    result['low30'] = result['index_low'].shift(1).rolling(window).min()

    result['action'] = np.nan
    result.loc[result['index_close'] > result['hi30'], 'action'] = 1.0
    result.loc[result['index_close'] < result['low30'], 'action'] = -1.0

    result['position'] = result['action'].ffill().fillna(0.0)
    result.loc[result[['hi30', 'low30']].isna().any(axis=1), 'position'] = np.nan

    result['strategy_ret'] = result['position'].shift(1) * result['ret']
    result['equity'] = (1 + result['strategy_ret'].fillna(0)).cumprod()

    return result

## Part 1: Compare systems on each instrument

In [4]:
asset_data = {
    'URO': uro,
    'TY': ty,
    'SP': sp,
}

asset_results = {}
summary_rows = []

for asset_name, df in asset_data.items():
    asset_results[asset_name] = {}

    asset_results[asset_name]['10/30'] = moving_average_strategy(df, 10, 30)
    asset_results[asset_name]['30/100'] = moving_average_strategy(df, 30, 100)
    asset_results[asset_name]['80/160'] = moving_average_strategy(df, 80, 160)
    asset_results[asset_name]['Breakout30'] = breakout_strategy(df, 30)

    for system_name, result_df in asset_results[asset_name].items():
        clean = result_df.dropna(subset=['strategy_ret']).copy()

        summary_rows.append({
            'asset': asset_name,
            'system': system_name,
            'start_date': clean['Date'].iloc[0],
            'end_date': clean['Date'].iloc[-1],
            'avg_daily_ret': clean['strategy_ret'].mean(),
            'daily_vol': clean['strategy_ret'].std(),
            'sharpe': sharpe_ratio(clean['strategy_ret']),
            'cum_return': (1 + clean['strategy_ret']).prod() - 1,
        })

summary = pd.DataFrame(summary_rows)
summary_display = summary.copy()

for col in ['avg_daily_ret', 'daily_vol', 'sharpe', 'cum_return']:
    summary_display[col] = summary_display[col].round(4)

summary_display

,asset,system,start_date,end_date,avg_daily_ret,daily_vol,sharpe,cum_return
0,URO,10/30,1999-04-19,2010-12-31,0.0002,0.0066,0.3885,0.5100
1,URO,30/100,1999-07-28,2010-12-31,0.0002,0.0067,0.4996,0.7101
2,URO,80/160,1999-10-22,2010-12-31,0.0001,0.0067,0.2695,0.2907
3,URO,Breakout30,1999-04-20,2010-12-31,0.0002,0.0066,0.4292,0.5869
4,TY,10/30,1999-04-19,2010-12-31,0.0001,0.0043,0.4376,0.3753
5,TY,30/100,1999-07-28,2010-12-31,0.0001,0.0043,0.3149,0.2425
6,TY,80/160,1999-10-22,2010-12-31,0.0000,0.0043,0.0096,-0.0184
7,TY,Breakout30,1999-04-20,2010-12-31,0.0001,0.0043,0.2134,0.1520
8,SP,10/30,1999-04-19,2010-12-31,-0.0000,0.0137,-0.0396,-0.3148
9,SP,30/100,1999-07-28,2010-12-31,-0.0000,0.0138,-0.0433,-0.3173


In [5]:
sharpe_table = summary.pivot(index='asset', columns='system', values='sharpe').round(3)
sharpe_table

system,10/30,30/100,80/160,Breakout30
asset,,,,
SP,-0.0400,-0.0430,0.1640,-0.3380
TY,0.4380,0.3150,0.0100,0.2130
URO,0.3890,0.5000,0.2700,0.4290


In [6]:
# Get best sharpe system for each asset
best_systems = (
    summary.sort_values(['asset', 'sharpe'], ascending=[True, False])
    .groupby('asset')
    .first()
    .reset_index()
)

best_systems_display = best_systems[['asset', 'system', 'start_date', 'sharpe', 'cum_return']].copy()
best_systems_display['sharpe'] = best_systems_display['sharpe'].round(4)
best_systems_display['cum_return'] = best_systems_display['cum_return'].round(4)
best_systems_display

,asset,system,start_date,sharpe,cum_return
0,SP,80/160,1999-10-21,0.1638,0.1410
1,TY,10/30,1999-04-19,0.4376,0.3753
2,URO,30/100,1999-07-28,0.4996,0.7101


## Part 2: Equal-weight combination of best sharpe system for each instrument

In [7]:
combo = None
# Compiling best strategy results for each asset
for asset_name in ['URO', 'TY', 'SP']:
    system_name = best_systems.loc[best_systems['asset'] == asset_name, 'system'].iloc[0]
    temp = asset_results[asset_name][system_name][['Date', 'strategy_ret']].copy()
    temp = temp.rename(columns={'strategy_ret': asset_name}) # Name best performing strategy for an asset after the asset

    if combo is None:
        combo = temp
    else:
        combo = combo.merge(temp, on='Date', how='inner')

combo = combo[combo['Date'] >= pd.Timestamp('1999-10-20')].copy()
combo['equal_weight_ret'] = combo[['URO', 'TY', 'SP']].mean(axis=1, skipna=False)
combo = combo.dropna(subset=['equal_weight_ret']).reset_index(drop=True)
combo['equal_weight_index'] = (1 + combo['equal_weight_ret']).cumprod()

equal_weight_sharpe = sharpe_ratio(combo['equal_weight_ret'])
largest_drawdown, peak_date, trough_date = max_drawdown(combo['equal_weight_index'], combo['Date'])

equal_weight_summary = pd.DataFrame({
    'portfolio': ['Equal weight'],
    'start_date': [combo['Date'].iloc[0]],
    'end_date': [combo['Date'].iloc[-1]],
    'sharpe': [equal_weight_sharpe],
    'largest_drawdown': [largest_drawdown],
    'peak_date': [peak_date],
    'trough_date': [trough_date],
})

equal_weight_summary_display = equal_weight_summary.copy()
equal_weight_summary_display['sharpe'] = equal_weight_summary_display['sharpe'].round(4)
equal_weight_summary_display['largest_drawdown'] = equal_weight_summary_display['largest_drawdown'].round(4)
equal_weight_summary_display

,portfolio,start_date,end_date,sharpe,largest_drawdown,peak_date,trough_date
0,Equal weight,1999-10-21,2010-12-31,0.4875,-0.1126,2001-09-21,2002-05-14


## Part 3: Inverse-volatility combination

In [8]:
for asset_name in ['URO', 'TY', 'SP']:
    combo['vol20_' + asset_name] = combo[asset_name].rolling(20).std()
    combo['raw_w_' + asset_name] = 1 / combo['vol20_' + asset_name]

combo['raw_weight_sum'] = combo[['raw_w_URO', 'raw_w_TY', 'raw_w_SP']].sum(axis=1)

for asset_name in ['URO', 'TY', 'SP']:
    combo['w_' + asset_name] = combo['raw_w_' + asset_name] / combo['raw_weight_sum']

combo['ivol_ret'] = (
    combo['w_URO'].shift(1) * combo['URO']
    + combo['w_TY'].shift(1) * combo['TY']
    + combo['w_SP'].shift(1) * combo['SP']
)
combo['ivol_index'] = (1 + combo['ivol_ret'].fillna(0)).cumprod()

ivol_summary = pd.DataFrame({
    'portfolio': ['Inverse vol'],
    'start_date': [combo.dropna(subset=['ivol_ret'])['Date'].iloc[0]],
    'end_date': [combo.dropna(subset=['ivol_ret'])['Date'].iloc[-1]],
    'sharpe': [sharpe_ratio(combo['ivol_ret'])],
})

ivol_summary_display = ivol_summary.copy()
ivol_summary_display['sharpe'] = ivol_summary_display['sharpe'].round(4)
ivol_summary_display

,portfolio,start_date,end_date,sharpe
0,Inverse vol,1999-11-18,2010-12-31,0.8450


In [9]:
print('Best system for URO:', best_systems.loc[best_systems['asset'] == 'URO', 'system'].iloc[0])
print('Best system for TY :', best_systems.loc[best_systems['asset'] == 'TY', 'system'].iloc[0])
print('Best system for SP :', best_systems.loc[best_systems['asset'] == 'SP', 'system'].iloc[0])
print()
print('Equal-weight portfolio Sharpe:', round(equal_weight_sharpe, 3))
print('Largest drawdown:', round(largest_drawdown, 4))
print('Drawdown peak date:', peak_date.date())
print('Drawdown trough date:', trough_date.date())
print('Inverse-vol portfolio Sharpe:', round(sharpe_ratio(combo['ivol_ret']), 3))

Best system for URO: 30/100
Best system for TY : 10/30
Best system for SP : 80/160

Equal-weight portfolio Sharpe: 0.487
Largest drawdown: -0.1126
Drawdown peak date: 2001-09-21
Drawdown trough date: 2002-05-14
Inverse-vol portfolio Sharpe: 0.845
